# KV Cache Compression Research Harness (Unified LongBench + Ruler Evaluation)

This notebook is designed for your use case:

- **No Ollama requirement** (Ollama backend is optional)
- Run KV-compression proxy methods in one framework: `fullkv`, `commonkv`, `minicache`, `thinkkv`, `palu`
- Run **LongBench + Ruler** style evaluation in one pass
- Produce a **single summary table** with grouped metrics like CommonKV's reporting format

> The compression methods below are prompt-level proxies. If you have true runtime KV implementations, plug them into the same `KVCompressionMethod` interface and keep the evaluation pipeline unchanged.

## 1) Setup

In [ ]:
# !pip install -U numpy pandas matplotlib tqdm requests datasets transformers accelerate sentencepiece

import math
import re
import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 2) Text utilities + metrics

In [ ]:
WORD_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def tokenize(text: str) -> List[str]:
    return WORD_RE.findall(text or "")


def detokenize(tokens: List[str]) -> str:
    out = []
    for t in tokens:
        if not out:
            out.append(t)
            continue
        prev = out[-1]
        if re.match(r"\w", t) and re.match(r".*\w$", prev):
            out.append(" " + t)
        elif t in ".,!?;:)]}" :
            out.append(t)
        elif prev.endswith("("):
            out.append(t)
        else:
            out.append(" " + t)
    return "".join(out)


def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?\n])\s+", (text or "").strip())
    return [p.strip() for p in parts if p.strip()]


def normalize_word_set(text: str) -> set:
    return set(t.lower() for t in tokenize(text) if re.match(r"\w+", t))


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    u = len(a | b)
    return (len(a & b) / u) if u else 0.0


def estimate_tokens(text: str) -> int:
    return max(1, math.ceil(len(text) / 4)) if text else 0


def lcs_len(a: List[str], b: List[str]) -> int:
    # O(nm), fine for short answers
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


def metric_exact_match(pred: str, refs: List[str]) -> float:
    p = (pred or "").strip().lower()
    return float(any(p == (r or "").strip().lower() for r in refs))


def metric_token_f1(pred: str, refs: List[str]) -> float:
    ptoks = [t.lower() for t in tokenize(pred) if re.match(r"\w+", t)]
    if not ptoks:
        return 0.0
    best = 0.0
    for r in refs:
        rtoks = [t.lower() for t in tokenize(r) if re.match(r"\w+", t)]
        if not rtoks:
            continue
        common = 0
        rcount = {}
        for t in rtoks:
            rcount[t] = rcount.get(t, 0) + 1
        for t in ptoks:
            if rcount.get(t, 0) > 0:
                common += 1
                rcount[t] -= 1
        if common == 0:
            continue
        p = common / len(ptoks)
        rr = common / len(rtoks)
        f1 = 2 * p * rr / (p + rr)
        best = max(best, f1)
    return best


def metric_rouge_l_f1(pred: str, refs: List[str]) -> float:
    pt = [t.lower() for t in tokenize(pred) if re.match(r"\w+", t)]
    if not pt:
        return 0.0
    best = 0.0
    for r in refs:
        rt = [t.lower() for t in tokenize(r) if re.match(r"\w+", t)]
        if not rt:
            continue
        l = lcs_len(pt, rt)
        if l == 0:
            continue
        p = l / len(pt)
        rr = l / len(rt)
        f = 2 * p * rr / (p + rr)
        best = max(best, f)
    return best


def metric_numeric_match(pred: str, refs: List[str]) -> float:
    nums = re.findall(r"-?\d+(?:\.\d+)?", pred or "")
    if not nums:
        return 0.0
    pset = set(nums)
    for r in refs:
        rset = set(re.findall(r"-?\d+(?:\.\d+)?", r or ""))
        if pset & rset:
            return 1.0
    return 0.0


def score_prediction(pred: str, refs: List[str], metric_name: str) -> float:
    metric_name = (metric_name or "token_f1").lower()
    if metric_name in {"em", "exact_match"}:
        return metric_exact_match(pred, refs)
    if metric_name in {"f1", "token_f1", "qa_f1"}:
        return metric_token_f1(pred, refs)
    if metric_name in {"rouge", "rouge_l", "rouge_l_f1", "summary_rouge"}:
        return metric_rouge_l_f1(pred, refs)
    if metric_name in {"numeric", "num", "numeric_match"}:
        return metric_numeric_match(pred, refs)
    return metric_token_f1(pred, refs)

## 3) KV compression methods (implemented proxies)

In [ ]:
class KVCompressionMethod:
    name = "base"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        return prompt


class FullKV(KVCompressionMethod):
    name = "fullkv"


class CommonKV(KVCompressionMethod):
    """Remove near-duplicate sentences and keep salient subset."""
    name = "commonkv"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sents = split_sentences(prompt)
        if len(sents) <= 3:
            return prompt

        sim_th = config.get("commonkv_similarity_threshold", 0.88)
        keep_budget = config.get("commonkv_keep_sentences", max(8, len(sents) // 2))

        uniq = []
        seen_sets = []
        for s in sents:
            sset = normalize_word_set(s)
            if any(jaccard(sset, prior) >= sim_th for prior in seen_sets):
                continue
            uniq.append(s)
            seen_sets.append(sset)

        scored = []
        for s in uniq:
            toks = [t for t in tokenize(s) if re.match(r"\w+", t)]
            if not toks:
                continue
            uniq_ratio = len(set(t.lower() for t in toks)) / len(toks)
            num_bonus = 0.2 if re.search(r"\d", s) else 0.0
            code_bonus = 0.2 if re.search(r"[{}();=<>]", s) else 0.0
            scored.append((uniq_ratio + num_bonus + code_bonus, s))

        scored.sort(key=lambda x: x[0], reverse=True)
        chosen = [s for _, s in scored[:keep_budget]]
        order = {s: i for i, s in enumerate(sents)}
        chosen.sort(key=lambda s: order.get(s, 10**9))
        return "\n".join(chosen)


class MiniCache(KVCompressionMethod):
    """Head/tail keep + periodic anchors across middle."""
    name = "minicache"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        toks = tokenize(prompt)
        n = len(toks)
        if n == 0:
            return prompt

        head = int(config.get("minicache_head_tokens", 700))
        tail = int(config.get("minicache_tail_tokens", 500))
        stride = int(config.get("minicache_anchor_stride", 400))
        width = int(config.get("minicache_anchor_width", 60))

        keep = set(range(min(head, n)))
        keep.update(range(max(0, n - tail), n))

        i = head
        upper = max(head, n - tail)
        while i < upper:
            keep.update(range(i, min(i + width, n)))
            i += max(width, stride)

        return detokenize([toks[i] for i in sorted(keep)])


class ThinkKV(KVCompressionMethod):
    """Importance-ranked sentence selection with boundary preservation."""
    name = "thinkkv"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sents = split_sentences(prompt)
        if len(sents) <= 4:
            return prompt

        head_keep = int(config.get("thinkkv_head_sentences", 2))
        tail_keep = int(config.get("thinkkv_tail_sentences", 2))
        budget = int(config.get("thinkkv_total_sentences", max(8, len(sents) // 3)))

        mid = sents[head_keep: max(head_keep, len(sents) - tail_keep)]
        scored = []
        for s in mid:
            toks = [t.lower() for t in tokenize(s) if re.match(r"\w+", t)]
            if not toks:
                continue
            uniq = len(set(toks)) / len(toks)
            len_bonus = min(len(toks) / 40.0, 1.0) * 0.4
            ent_bonus = 0.3 if re.search(r"[A-Z][a-z]+", s) else 0.0
            num_bonus = 0.3 if re.search(r"\d", s) else 0.0
            scored.append((uniq + len_bonus + ent_bonus + num_bonus, s))

        need = max(0, budget - head_keep - tail_keep)
        scored.sort(key=lambda x: x[0], reverse=True)
        chosen_mid = [s for _, s in scored[:need]]
        order = {s: i for i, s in enumerate(sents)}
        chosen_mid.sort(key=lambda s: order[s])

        return "\n".join(sents[:head_keep] + chosen_mid + sents[-tail_keep:])


def _hash_embed(text: str, dim: int = 128) -> np.ndarray:
    vec = np.zeros(dim, dtype=float)
    for t in tokenize(text):
        if not re.match(r"\w+", t):
            continue
        vec[hash(t.lower()) % dim] += 1.0
    n = np.linalg.norm(vec)
    return vec / n if n > 0 else vec


class PaLu(KVCompressionMethod):
    """Cluster sentence embeddings and keep representatives."""
    name = "palu"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        sents = split_sentences(prompt)
        if len(sents) <= 6:
            return prompt

        k = int(config.get("palu_clusters", min(12, max(4, len(sents) // 12))))
        dim = int(config.get("palu_embed_dim", 128))
        iters = int(config.get("palu_kmeans_iters", 6))

        X = np.vstack([_hash_embed(s, dim=dim) for s in sents])
        rng = np.random.default_rng(int(config.get("seed", 42)))
        init = rng.choice(len(sents), size=min(k, len(sents)), replace=False)
        C = X[init].copy()

        for _ in range(iters):
            sims = X @ C.T
            labels = sims.argmax(axis=1)
            for ci in range(C.shape[0]):
                mem = X[labels == ci]
                if len(mem) == 0:
                    continue
                c = mem.mean(axis=0)
                n = np.linalg.norm(c)
                C[ci] = c / n if n > 0 else c

        sims = X @ C.T
        labels = sims.argmax(axis=1)

        keep = {0, len(sents) - 1}
        for ci in range(C.shape[0]):
            idx = np.where(labels == ci)[0]
            if len(idx) == 0:
                continue
            best = idx[(X[idx] @ C[ci]).argmax()]
            keep.add(int(best))

        return "\n".join(sents[i] for i in sorted(keep))


METHODS = {
    "fullkv": FullKV(),
    "commonkv": CommonKV(),
    "minicache": MiniCache(),
    "minichache": MiniCache(),
    "thinkkv": ThinkKV(),
    "thinkk": ThinkKV(),
    "palu": PaLu(),
}

print("Methods:", sorted(METHODS.keys()))

## 4) Backends (HF local model default; Ollama optional)

In [ ]:
class GenerationBackend:
    def generate(self, model: str, prompt: str, generation_config: Dict[str, Any]) -> str:
        raise NotImplementedError


class HFBackend(GenerationBackend):
    """Transformers backend; preferred when you do not want Ollama."""

    def __init__(self, device_map: str = "auto"):
        self.device_map = device_map
        self._pipes = {}

    def _load_pipe(self, model: str):
        if model in self._pipes:
            return self._pipes[model]
        from transformers import pipeline
        gen = pipeline("text-generation", model=model, device_map=self.device_map)
        self._pipes[model] = gen
        return gen

    def generate(self, model: str, prompt: str, generation_config: Dict[str, Any]) -> str:
        gen = self._load_pipe(model)
        max_new = int(generation_config.get("max_new_tokens", 256))
        temp = float(generation_config.get("temperature", 0.0))
        do_sample = temp > 0
        out = gen(prompt, max_new_tokens=max_new, do_sample=do_sample, temperature=max(temp, 1e-5) if do_sample else None)
        if not out:
            return ""
        txt = out[0].get("generated_text", "")
        # most pipelines return prompt + completion
        return txt[len(prompt):] if txt.startswith(prompt) else txt


class OllamaBackend(GenerationBackend):
    """Optional backend with robust endpoint fallback."""

    def __init__(self, base_url: str = "http://localhost:11434", timeout_s: int = 300):
        import requests
        self.requests = requests
        self.base_url = base_url.rstrip("/")
        self.timeout_s = timeout_s

    def _post(self, path: str, payload: Dict[str, Any]):
        return self.requests.post(f"{self.base_url}{path}", json=payload, timeout=self.timeout_s)

    def generate(self, model: str, prompt: str, generation_config: Dict[str, Any]) -> str:
        payload = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": generation_config.get("temperature", 0.0),
                "num_predict": generation_config.get("max_new_tokens", 256),
                "num_ctx": generation_config.get("num_ctx", 8192),
            },
        }

        # 1) /api/generate
        r = self._post("/api/generate", payload)
        if r.ok:
            return r.json().get("response", "")
        if r.status_code not in (404, 405):
            r.raise_for_status()

        # 2) /api/chat
        r = self._post("/api/chat", {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": payload["options"],
        })
        if r.ok:
            return r.json().get("message", {}).get("content", "")
        if r.status_code not in (404, 405):
            r.raise_for_status()

        # 3) /v1/chat/completions
        r = self._post("/v1/chat/completions", {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": payload["options"]["temperature"],
            "max_tokens": payload["options"]["num_predict"],
        })
        r.raise_for_status()
        return r.json().get("choices", [{}])[0].get("message", {}).get("content", "")

## 5) Benchmark example format + loaders (LongBench/Ruler)

In [ ]:
@dataclass
class BenchmarkExample:
    example_id: str
    benchmark: str       # longbench or ruler
    group: str           # e.g., QA, Sum, FShot, Synth, Code, NS...
    task: str            # dataset/task name
    metric: str          # em/f1/rouge_l/numeric
    prompt: str
    references: List[str]
    meta: Dict[str, Any] = field(default_factory=dict)


def _first_present(d: Dict[str, Any], keys: List[str], default=""):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default


def _as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v) for v in x if v is not None]
    return [str(x)]


def load_longbench_from_hf(task_specs: List[Dict[str, str]], split: str = "test", limit_per_task: Optional[int] = None) -> List[BenchmarkExample]:
    """Load LongBench tasks from HF datasets.

    task_specs item example:
      {"task": "narrativeqa", "group": "QA", "metric": "f1"}
    """
    from datasets import load_dataset

    rows = []
    for spec in task_specs:
        task = spec["task"]
        ds = load_dataset("THUDM/LongBench", task, split=split)
        if limit_per_task:
            ds = ds.select(range(min(limit_per_task, len(ds))))

        for i, ex in enumerate(ds):
            prompt = _first_present(ex, ["prompt", "input", "context", "question"], "")
            if "context" in ex and "question" in ex and "prompt" not in ex:
                prompt = f"Context:
{ex.get('context','')}

Question: {ex.get('question','')}
Answer:"

            refs = _as_list(_first_present(ex, ["answers", "answer", "output", "target"], []))
            rows.append(BenchmarkExample(
                example_id=f"longbench::{task}::{i}",
                benchmark="longbench",
                group=spec["group"],
                task=task,
                metric=spec.get("metric", "f1"),
                prompt=str(prompt),
                references=refs if refs else [""],
                meta={"raw_keys": list(ex.keys())},
            ))
    return rows


def load_ruler_from_jsonl(root_dir: str, task_specs: List[Dict[str, str]], limit_per_task: Optional[int] = None) -> List[BenchmarkExample]:
    """Load Ruler tasks from local JSONL files.

    Expects files like: <root_dir>/<task>.jsonl
    Each line should include at minimum prompt/input and answer(s).
    """
    rows = []
    root = Path(root_dir)

    for spec in task_specs:
        task = spec["task"]
        fp = root / f"{task}.jsonl"
        if not fp.exists():
            print(f"[WARN] Missing Ruler file: {fp}")
            continue

        with fp.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if limit_per_task and i >= limit_per_task:
                    break
                line = line.strip()
                if not line:
                    continue
                ex = json.loads(line)
                prompt = _first_present(ex, ["prompt", "input", "context", "question"], "")
                refs = _as_list(_first_present(ex, ["answers", "answer", "output", "target"], []))
                rows.append(BenchmarkExample(
                    example_id=f"ruler::{task}::{i}",
                    benchmark="ruler",
                    group=spec["group"],
                    task=task,
                    metric=spec.get("metric", "em"),
                    prompt=str(prompt),
                    references=refs if refs else [""],
                    meta={},
                ))
    return rows

## 6) Unified runner (compression + generation + scoring in one pass)

In [ ]:
def run_benchmark(
    examples: List[BenchmarkExample],
    methods: List[str],
    model: str,
    backend: GenerationBackend,
    compression_config: Dict[str, Any],
    generation_config: Dict[str, Any],
) -> pd.DataFrame:
    rows = []

    total = len(examples) * len(methods)
    with tqdm(total=total, desc="Benchmark") as pbar:
        for ex in examples:
            for m in methods:
                method = METHODS[m]

                compressed_prompt = method.preprocess_prompt(ex.prompt, compression_config)
                t0 = time.perf_counter()
                try:
                    pred = backend.generate(model=model, prompt=compressed_prompt, generation_config=generation_config)
                    err = ""
                except Exception as e:
                    pred = ""
                    err = str(e)
                latency = time.perf_counter() - t0

                score = 0.0 if err else score_prediction(pred, ex.references, ex.metric)

                rows.append({
                    "example_id": ex.example_id,
                    "benchmark": ex.benchmark,
                    "group": ex.group,
                    "task": ex.task,
                    "metric": ex.metric,
                    "method": m,
                    "model": model,
                    "orig_chars": len(ex.prompt),
                    "comp_chars": len(compressed_prompt),
                    "compression_ratio": (len(compressed_prompt) / len(ex.prompt)) if len(ex.prompt) else 1.0,
                    "orig_toks_est": estimate_tokens(ex.prompt),
                    "comp_toks_est": estimate_tokens(compressed_prompt),
                    "latency_s": latency,
                    "prediction": pred,
                    "score": score,
                    "error": err,
                })
                pbar.update(1)

    return pd.DataFrame(rows)

## 7) Task mapping to CommonKV-style reporting groups

In [ ]:
# Edit these lists to match exactly the task subsets you want.
# Group names chosen to mirror your requested table layout.

LONGBENCH_TASK_SPECS = [
    # group labels: QA, Sum., FShot, Synth., Code
    {"task": "narrativeqa", "group": "QA", "metric": "f1"},
    {"task": "qasper", "group": "QA", "metric": "f1"},
    {"task": "gov_report", "group": "Sum.", "metric": "rouge_l"},
    {"task": "multi_news", "group": "Sum.", "metric": "rouge_l"},
    {"task": "trec", "group": "FShot", "metric": "em"},
    {"task": "hotpotqa", "group": "Synth.", "metric": "f1"},
    {"task": "lcc", "group": "Code", "metric": "em"},
]

RULER_TASK_SPECS = [
    # group labels: NS, NMK, NMQ, NMV, QA, VT, FWE
    {"task": "ns", "group": "NS", "metric": "em"},
    {"task": "nmk", "group": "NMK", "metric": "em"},
    {"task": "nmq", "group": "NMQ", "metric": "em"},
    {"task": "nmv", "group": "NMV", "metric": "em"},
    {"task": "qa", "group": "QA", "metric": "f1"},
    {"task": "vt", "group": "VT", "metric": "em"},
    {"task": "fwe", "group": "FWE", "metric": "em"},
]

## 8) Load evaluation sets (LongBench + Ruler)

In [ ]:
# Option A: LongBench from HF + Ruler from local JSONL folder
# Option B: replace loaders with your local copies for both

LONG_LIMIT = 50      # set None for full evaluation
RULER_LIMIT = 50     # set None for full evaluation
RULER_ROOT = "./ruler_data"  # put ns.jsonl, nmk.jsonl, ... here

examples_long = load_longbench_from_hf(
    task_specs=LONGBENCH_TASK_SPECS,
    split="test",
    limit_per_task=LONG_LIMIT,
)
examples_ruler = load_ruler_from_jsonl(
    root_dir=RULER_ROOT,
    task_specs=RULER_TASK_SPECS,
    limit_per_task=RULER_LIMIT,
)

examples_all = examples_long + examples_ruler
print(f"Loaded examples: longbench={len(examples_long)}, ruler={len(examples_ruler)}, total={len(examples_all)}")

## 9) Run all methods once and build one table

In [ ]:
# Choose backend:
# backend = OllamaBackend(base_url="http://localhost:11434")
backend = HFBackend(device_map="auto")

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # replace with local/HF model you want
METHOD_LIST = ["fullkv", "commonkv", "minicache", "thinkkv", "palu"]

COMPRESSION_CONFIG = {
    "seed": 42,
    "commonkv_similarity_threshold": 0.88,
    "commonkv_keep_sentences": 140,
    "minicache_head_tokens": 700,
    "minicache_tail_tokens": 500,
    "minicache_anchor_stride": 400,
    "minicache_anchor_width": 60,
    "thinkkv_head_sentences": 2,
    "thinkkv_tail_sentences": 2,
    "thinkkv_total_sentences": 120,
    "palu_clusters": 10,
    "palu_embed_dim": 128,
    "palu_kmeans_iters": 6,
}

GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.0,
    "num_ctx": 8192,
}

results_df = run_benchmark(
    examples=examples_all,
    methods=METHOD_LIST,
    model=MODEL_NAME,
    backend=backend,
    compression_config=COMPRESSION_CONFIG,
    generation_config=GEN_CONFIG,
)

results_df.head(5)

## 10) One CommonKV-style final table (all evals together)

In [ ]:
def build_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    # per benchmark/group/method score means
    agg = (
        results_df
        .groupby(["benchmark", "group", "method"], as_index=False)
        .agg(score=("score", "mean"),
             latency=("latency_s", "mean"),
             compression=("compression_ratio", "mean"))
    )

    # pivot scores for one-table report
    piv = agg.pivot_table(index="method", columns=["benchmark", "group"], values="score", aggfunc="mean")

    # flatten column names
    piv.columns = [f"{b}:{g}" for b, g in piv.columns]

    # global averages
    method_avg = results_df.groupby("method", as_index=True)["score"].mean().rename("Avg.")
    out = piv.join(method_avg, how="left")

    # attach latency/compression side metrics
    side = results_df.groupby("method", as_index=True).agg(
        latency_s=("latency_s", "mean"),
        compression_ratio=("compression_ratio", "mean"),
        rows=("score", "count"),
    )
    out = out.join(side, how="left")

    return out.sort_values("Avg.", ascending=False)


summary_table = build_summary_table(results_df)
summary_table

## 11) Export results and summary

In [ ]:
out_dir = Path("artifacts")
out_dir.mkdir(parents=True, exist_ok=True)

raw_path = out_dir / "kv_longbench_ruler_raw.csv"
summary_path = out_dir / "kv_longbench_ruler_summary.csv"

results_df.to_csv(raw_path, index=False)
summary_table.to_csv(summary_path)

print("Saved:")
print(" -", raw_path)
print(" -", summary_path)

## 12) Notes for your next step

- If you implement true KV-compression kernels (instead of prompt proxies), keep this notebook's evaluation code and only swap method internals.
- You can reproduce the CommonKV-style table format by adjusting `LONGBENCH_TASK_SPECS` and `RULER_TASK_SPECS` task lists/metrics to match their exact paper split.
- Since you requested unified evaluation, all scores are produced by one runner and one final table (`summary_table`) without separate scripts.